In [1]:
# End-to-end Manim video generation for first 3 topics
#
# Flow:
# 1) Read first 3 valid topics from /Users/alyssa/Desktop/education_video/topics.xlsx
# 2) Use OpenRouter (model: openai/gpt-5.5) to generate Manim scene code
# 3) Render each topic to MP4 via manim CLI
# 4) Save outputs under /Users/alyssa/Desktop/education_video/manim_outputs

In [19]:
# 1. 更新系统源，并安装 manim 编译所需的全部 Linux 底层开发库
!apt-get update && apt-get install -y \
    libpango1.0-dev \
    pkg-config \
    ffmpeg \
    freeglut3-dev

# 2. 补齐系统依赖后，再执行强制全局安装，此时 pip 就能顺利完成本地 wheel 编译
!python3 -m pip install -U --break-system-packages manim

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,083 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,308 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,332 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:13

In [20]:
# 1) Install runtime dependencies (run once)
# Colab: install into current interpreter directly with apt system dependencies.
# Local: use a dedicated venv when possible.

import sys
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_ROOT = Path("/content/education_video") if IN_COLAB else Path("/Users/alyssa/Desktop/education_video")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    print("--> Detecting Google Colab environment. Preparing system dependencies...")
    
    # 1. 核心修复：先用 apt-get 补齐 manimpango 编译依赖，防止 pip 罢工
    try:
        subprocess.check_call(["apt-get", "update", "-qq"])
        subprocess.check_call(["apt-get", "install", "-y", "-qq", "libpango1.0-dev", "pkg-config", "ffmpeg"])
        print("System dependencies (Pango, FFmpeg) installed successfully.")
    except Exception as e:
        print(f"Warning: System apt install failed ({e}). Proceeding to pip anyway.")

    # 2. 核心修复：加上 '--break-system-packages' 绕过 Python 3.12 保护锁
    PIP_BASE = [sys.executable, "-m", "pip", "install", "-U", "--break-system-packages"]
    MANIM_PYTHON = Path(sys.executable)

    print("--> Installing Python dependencies via pip...")
    subprocess.check_call(PIP_BASE + ["openai", "openpyxl", "json-repair"])

    local_manim = WORK_ROOT / "manim"
    if local_manim.exists():
        try:
            subprocess.check_call(PIP_BASE + ["-e", str(local_manim)])
            print("Installed local manim repo in editable mode (colab).")
        except subprocess.CalledProcessError as e:
            print(f"Local editable install failed ({e}). Falling back to PyPI manim.")
            subprocess.check_call(PIP_BASE + ["manim"])
    else:
        # 去掉 ==0.20.1，允许拉取最新的稳定编译版本
        subprocess.check_call(PIP_BASE + ["manim"])

else:
    print("--> Detecting Local environment. Setting up isolated venv...")
    # Local notebook (PEP668-safe): user packages + optional venv for manim
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--user",
        "--break-system-packages",
        "-U",
        "openai",
        "openpyxl",
        "json-repair",
    ])

    VENV_DIR = WORK_ROOT / ".venv_manim_runner"
    MANIM_PYTHON = VENV_DIR / "bin" / "python"

    if not MANIM_PYTHON.exists():
        try:
            subprocess.check_call([sys.executable, "-m", "venv", str(VENV_DIR)])
        except subprocess.CalledProcessError:
            # Fallback to current interpreter if venv is unavailable
            MANIM_PYTHON = Path(sys.executable)

    subprocess.check_call([str(MANIM_PYTHON), "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])

    local_manim = WORK_ROOT / "manim"
    if local_manim.exists():
        try:
            subprocess.check_call([str(MANIM_PYTHON), "-m", "pip", "install", "-e", str(local_manim)])
            print("Installed local manim repo in editable mode.")
        except subprocess.CalledProcessError as e:
            print(f"Local editable install failed ({e}). Falling back to PyPI manim.")
            subprocess.check_call([str(MANIM_PYTHON), "-m", "pip", "install", "manim==0.20.1"])
    else:
        subprocess.check_call([str(MANIM_PYTHON), "-m", "pip", "install", "manim==0.20.1"])

# Validate manim module in selected interpreter
print("--> Validating Manim environment...")
subprocess.check_call([str(MANIM_PYTHON), "-c", "import manim; print('Manim version:', manim.__version__)"])

print("\n" + "="*40)
print("Work root:", WORK_ROOT)
print("Manim python:", MANIM_PYTHON)
print("Dependencies successfully installed!")
print("="*40)

--> Detecting Google Colab environment. Preparing system dependencies...
System dependencies (Pango, FFmpeg) installed successfully.
--> Installing Python dependencies via pip...
--> Validating Manim environment...

Work root: /content/education_video
Manim python: /usr/bin/python3
Dependencies successfully installed!


In [21]:
# 2) Configuration (+ Google Drive integration)

import os
import sys
import shutil
from pathlib import Path
from datetime import datetime

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from colab_bootstrap import ensure_openai_key, setup_api_tracking, openai_model, print_api_usage

ensure_openai_key()
setup_api_tracking()
OPENAI_MODEL = openai_model()
MAX_TOPICS = 0  # 0 = all remaining topics
QUALITY_FLAG = "-ql"  # -ql quick, -qm medium, -qh high
MAX_REVIEWER_FIXES = 3  # max automatic syntax-fix retries per topic

IN_COLAB = False
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DRIVE_INPUT_DIR = None
DRIVE_OUTPUT_ROOT = None

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    WORK_ROOT = Path("/content/education_video")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)

    DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/education_video")
    DRIVE_OUTPUT_ROOT = DRIVE_INPUT_DIR / "manim_outputs"
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    # If project folder is available on Drive, sync it locally (optional but preferred).
    drive_manim = DRIVE_INPUT_DIR / "manim"
    local_manim = WORK_ROOT / "manim"
    if drive_manim.exists() and not local_manim.exists():
        shutil.copytree(drive_manim, local_manim)

    # Always use topics.xlsx from Drive in Colab.
    TOPICS_XLSX = DRIVE_INPUT_DIR / "topics.xlsx"
    OUTPUT_ROOT = WORK_ROOT / "manim_outputs" / datetime.now().strftime("run_%Y%m%d_%H%M%S")
else:
    WORK_ROOT = Path("/Users/alyssa/Desktop/education_video")

    # Local Google Drive Desktop candidates (macOS)
    # Optional manual override:
    # export MANIM_DRIVE_EDU_DIR="/path/to/GoogleDrive/.../My Drive/education_video"
    manual_drive_dir = os.getenv("MANIM_DRIVE_EDU_DIR", "").strip()
    if manual_drive_dir:
        p = Path(manual_drive_dir)
        if p.exists():
            DRIVE_INPUT_DIR = p
            DRIVE_OUTPUT_ROOT = p / "manim_outputs"
            DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    if DRIVE_INPUT_DIR is None:
        candidates = []
        cloud_root = Path.home() / "Library" / "CloudStorage"
        if cloud_root.exists():
            candidates.extend(cloud_root.glob("GoogleDrive*"))
        for base in candidates:
            edu = base / "My Drive" / "education_video"
            if edu.exists():
                DRIVE_INPUT_DIR = edu
                DRIVE_OUTPUT_ROOT = edu / "manim_outputs"
                DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
                break

    TOPICS_XLSX = WORK_ROOT / "topics.xlsx"
    if not TOPICS_XLSX.exists() and DRIVE_INPUT_DIR is not None:
        drive_topics = DRIVE_INPUT_DIR / "topics.xlsx"
        if drive_topics.exists():
            TOPICS_XLSX = drive_topics

    OUTPUT_ROOT = WORK_ROOT / "manim_outputs" / datetime.now().strftime("run_%Y%m%d_%H%M%S")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Recompute Manim interpreter for CURRENT environment each run.
VENV_DIR = WORK_ROOT / ".venv_manim_runner"
candidate_manim_python = VENV_DIR / "bin" / "python"
if candidate_manim_python.exists():
    MANIM_PYTHON = candidate_manim_python
else:
    MANIM_PYTHON = Path(sys.executable)

print("Model:", OPENAI_MODEL)
print("In Colab:", IN_COLAB)
print("Work root:", WORK_ROOT)
print("Topics file:", TOPICS_XLSX)
print("Output root:", OUTPUT_ROOT)
print("Manim python:", MANIM_PYTHON)
if DRIVE_INPUT_DIR is not None:
    print("Drive input dir:", DRIVE_INPUT_DIR)
if DRIVE_OUTPUT_ROOT is not None:
    print("Drive output root:", DRIVE_OUTPUT_ROOT)
else:
    print("Drive output root: not found (local Google Drive folder not detected)")
print("Max topics:", MAX_TOPICS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model: openai/gpt-5.5
In Colab: True
Work root: /content/education_video
Topics file: /content/drive/MyDrive/education_video/topics.xlsx
Output root: /content/education_video/manim_outputs/run_20260707_083843
Manim python: /usr/bin/python3
Drive input dir: /content/drive/MyDrive/education_video
Drive output root: /content/drive/MyDrive/education_video/manim_outputs
Max topics: 3


In [22]:
# 3) Load the first 3 valid topics from the first worksheet

from openpyxl import load_workbook


def _clean(v):
    return "" if v is None else str(v).strip()


wb = load_workbook(TOPICS_XLSX, read_only=True, data_only=True)
ws = wb.worksheets[0]
rows = ws.iter_rows(values_only=True)
header = [_clean(x) for x in next(rows)]

topic_rows = []
for i, row in enumerate(rows, start=2):
    item = {header[idx]: _clean(row[idx]) if idx < len(row) else "" for idx in range(len(header))}
    req_id = item.get("Request ID", "")
    topic = item.get("Topic", "")
    persona = item.get("Student Persona", "")

    if not req_id or not topic:
        continue
    if req_id == "Request ID" or topic == "Topic":
        continue

    topic_rows.append(
        {
            "row": i,
            "request_id": req_id,
            "topic": topic,
            "persona": persona,
        }
    )
    if MAX_TOPICS > 0 and len(topic_rows) >= MAX_TOPICS:
        break

if not topic_rows:
    raise RuntimeError("No valid topics found in first worksheet.")

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from common_topics import topic_has_video

_output_check = DRIVE_OUTPUT_ROOT if DRIVE_OUTPUT_ROOT is not None else WORK_ROOT / "manim_outputs"
_pending = []
for t in topic_rows:
    if topic_has_video(_output_check, t["request_id"]):
        print(f"SKIP (done): [{t['request_id']}] {t['topic']}")
        continue
    _pending.append(t)
topic_rows = _pending[:MAX_TOPICS] if MAX_TOPICS > 0 else _pending

if not topic_rows:
    print("All manim topics already have videos.")
else:
    print("Selected topics:")
    for t in topic_rows:
        print(f"- [{t['request_id']}] {t['topic']} (row {t['row']})")

Selected topics:
- [tm_cs_bin_001] Base-2 Number Systems and Data Representation (row 3)
- [tm_cs_saf_002] Fundamentals of Personal Cybersecurity and Defense (row 4)
- [tm_cs_col_003] Version Control Principles via Collaborative Workflows (row 5)


In [23]:
# 4) Helpers: OpenRouter call, code extraction, and rendering

import json
import re
import subprocess
import sys
from openai import OpenAI
from json_repair import repair_json

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def safe_name(text: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_\- ]+", "", text).strip()
    s = re.sub(r"\s+", "_", s)
    return s[:80] if s else "topic"


def scene_class_name(topic: str) -> str:
    parts = re.findall(r"[A-Za-z0-9]+", topic)
    name = "".join(p.capitalize() for p in parts[:8]) or "Topic"
    if not name[0].isalpha():
        name = "Topic" + name
    return f"{name}Scene"


def build_prompt(topic: str, persona: str, class_name: str) -> str:
    persona_line = persona if persona else "General learners"
    return f"""
You are an expert Manim (Community v0.19+) educator.
Create ONE complete Python file that defines exactly one class `{class_name}(Scene)` and renders a clear educational animation.

Topic: {topic}
Target student persona: {persona_line}

Hard requirements:
1) Output code only, no markdown fences.
2) Must include: `from manim import *`.
3) Scene should be around 45-90 seconds at normal speed.
4) Use clear visual hierarchy, readable font sizes, good spacing.
5) Avoid external assets, files, web downloads, and latex packages beyond default Manim support.
6) Keep runtime stable on CPU.
7) No interactive input, no undefined symbols.
8) End with a concise recap.

Return only valid Python code.
""".strip()


def request_scene_code(topic: str, persona: str, class_name: str) -> str:
    prompt = build_prompt(topic, persona, class_name)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=6000,
        extra_headers={
            "HTTP-Referer": "https://localhost",
            "X-OpenRouter-Title": "Manim Topic Runner",
        },
    )
    content = (resp.choices[0].message.content or "").strip()
    if content.startswith("```"):
        content = re.sub(r"^```[a-zA-Z]*\n", "", content)
        content = re.sub(r"\n```$", "", content)
    return content.strip()


def render_scene(py_file: str, class_name: str, out_dir: str) -> None:
    python_bin = str(globals().get("MANIM_PYTHON", Path(sys.executable)))
    cmd = [
        python_bin,
        "-m",
        "manim",
        QUALITY_FLAG,
        py_file,
        class_name,
        "--media_dir",
        out_dir,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        details = (
            f"Manim render failed (exit_code={proc.returncode})\n"
            f"CMD: {' '.join(cmd)}\n\n"
            f"STDOUT:\n{proc.stdout}\n\n"
            f"STDERR:\n{proc.stderr}"
        )
        raise RuntimeError(details)

print("Helpers ready.")

Helpers ready.


In [24]:
# 5) End-to-end run with reviewer loop (generate -> render -> repair)

import ast
import traceback


def _extract_class_name(code: str) -> str | None:
    m = re.search(r"class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(\s*Scene\s*\)", code)
    return m.group(1) if m else None


def _repair_code_with_llm(code: str, error_text: str, topic: str, class_name: str) -> str:
    prompt = f"""
You are fixing Python syntax in a Manim script.

Topic: {topic}
Required class name: {class_name}

Rules:
1) Return ONLY valid Python code (no markdown fences).
2) Preserve behavior and structure as much as possible.
3) Fix syntax/runtime-breaking issues (missing quotes, parentheses, bad escapes, malformed calls, unmatched brackets).
4) Keep imports and class name intact.

Current code:
{code}

Render/parse error:
{error_text}
""".strip()

    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=7000,
        extra_headers={
            "HTTP-Referer": "https://localhost",
            "X-OpenRouter-Title": "Manim Reviewer Loop",
        },
    )
    fixed = (resp.choices[0].message.content or "").strip()
    if fixed.startswith("```"):
        fixed = re.sub(r"^```[a-zA-Z]*\n", "", fixed)
        fixed = re.sub(r"\n```$", "", fixed)
    return fixed.strip()


results = []

for idx, item in enumerate(topic_rows, start=1):
    topic = item["topic"]
    persona = item["persona"]
    req_id = item["request_id"]

    folder_name = f"{idx:02d}_{req_id}_{safe_name(topic)}"
    topic_dir = OUTPUT_ROOT / folder_name
    topic_dir.mkdir(parents=True, exist_ok=True)

    py_path = topic_dir / "scene.py"
    class_name = scene_class_name(topic)

    print("\n" + "=" * 80)
    print(f"[{idx}/{len(topic_rows)}] {topic}")
    print("Class:", class_name)

    code = request_scene_code(topic, persona, class_name)
    py_path.write_text(code, encoding="utf-8")

    fixed_count = 0
    success = False
    last_error = ""

    while fixed_count <= MAX_REVIEWER_FIXES:
        try:
            current_code = py_path.read_text(encoding="utf-8")
            try:
                ast.parse(current_code)
            except SyntaxError as se:
                raise RuntimeError(f"Python parse failed before render: {se}")

            render_scene(str(py_path), class_name, str(topic_dir))
            media_dir = topic_dir / "videos"
            mp4s = sorted(media_dir.rglob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
            video_path = str(mp4s[0]) if mp4s else ""

            results.append(
                {
                    "request_id": req_id,
                    "topic": topic,
                    "status": "ok",
                    "video": video_path,
                    "folder": str(topic_dir),
                    "fixes_used": fixed_count,
                }
            )
            print("Rendered:", video_path or "(mp4 not found)")
            success = True
            break

        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"
            (topic_dir / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")

            if fixed_count >= MAX_REVIEWER_FIXES:
                break

            fixed_count += 1
            raw_code = py_path.read_text(encoding="utf-8")
            detected_class = _extract_class_name(raw_code) or class_name
            repaired_code = _repair_code_with_llm(raw_code, last_error, topic, detected_class)
            py_path.write_text(repaired_code, encoding="utf-8")
            print(f"Repair attempt {fixed_count}/{MAX_REVIEWER_FIXES} for {topic}")

    if not success:
        results.append(
            {
                "request_id": req_id,
                "topic": topic,
                "status": "failed",
                "video": "",
                "folder": str(topic_dir),
                "error": last_error,
                "fixes_used": fixed_count,
            }
        )
        print("FAILED:", last_error)
        print("Wrote detailed log:", topic_dir / "error.txt")

summary_path = OUTPUT_ROOT / "summary.json"
summary_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nSaved summary:", summary_path)

# Auto-sync to Google Drive when configured
if globals().get("DRIVE_OUTPUT_ROOT") is not None:
    drive_run_dir = DRIVE_OUTPUT_ROOT / OUTPUT_ROOT.name
    if drive_run_dir.exists():
        shutil.rmtree(drive_run_dir)
    shutil.copytree(OUTPUT_ROOT, drive_run_dir)

    zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT))
    shutil.copy2(zip_path, DRIVE_OUTPUT_ROOT / zip_path.name)

    print("Synced folder to Drive:", drive_run_dir)
    print("Synced zip to Drive:", DRIVE_OUTPUT_ROOT / zip_path.name)


[1/3] Base-2 Number Systems and Data Representation
Class: Base2NumberSystemsAndDataRepresentationScene
FAILED: RuntimeError: Manim render failed (exit_code=1)
CMD: /usr/bin/python3 -m manim -ql /content/education_video/manim_outputs/run_20260707_083843/01_tm_cs_bin_001_Base-2_Number_Systems_and_Data_Representation/scene.py Base2NumberSystemsAndDataRepresentationScene --media_dir /content/education_video/manim_outputs/run_20260707_083843/01_tm_cs_bin_001_Base-2_Number_Systems_and_Data_Representation

STDOUT:
Manim Community v0.20.1



STDERR:
<frozen runpy>:128: RuntimeWarning: 'manim.__main__' found in sys.modules after import of package 'manim', but prior to execution of 'manim.__main__'; this may result in unpredictable behaviour
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/manim/__main__.py", line 101, in <module>
    main()
  File "/usr/lo

In [25]:
# 7) Repair failed scene.py files and continue rendering (for current OUTPUT_ROOT)

import ast
import re
import traceback
from pathlib import Path


def _extract_class_name(code: str) -> str | None:
    m = re.search(r"class\s+([A-Za-z_][A-Za-z0-9_]*)\s*\(\s*Scene\s*\)", code)
    return m.group(1) if m else None


def _repair_code_with_llm(code: str, error_text: str, topic: str, class_name: str) -> str:
    prompt = f"""
You are fixing Python syntax in a Manim script.

Topic: {topic}
Required class name: {class_name}

Rules:
1) Return ONLY valid Python code (no markdown fences).
2) Preserve behavior and structure as much as possible.
3) Fix ONLY syntax errors (missing quotes, parentheses, brackets, commas, escapes, etc.).
4) Keep imports and class name intact.

Current code:
{code}

Render error:
{error_text}
""".strip()

    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=7000,
        extra_headers={
            "HTTP-Referer": "https://localhost",
            "X-OpenRouter-Title": "Manim Syntax Repair",
        },
    )
    fixed = (resp.choices[0].message.content or "").strip()
    if fixed.startswith("```"):
        fixed = re.sub(r"^```[a-zA-Z]*\n", "", fixed)
        fixed = re.sub(r"\n```$", "", fixed)
    return fixed.strip()


def _is_valid_python(code: str) -> tuple[bool, str]:
    try:
        ast.parse(code)
        return True, ""
    except SyntaxError as e:
        return False, f"SyntaxError: {e}"


summary_path = OUTPUT_ROOT / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"No summary found: {summary_path}")

items = json.loads(summary_path.read_text(encoding="utf-8"))
failed_items = [x for x in items if x.get("status") != "ok"]

if not failed_items:
    print("No failed items in summary. Nothing to repair.")
else:
    print(f"Trying to repair {len(failed_items)} failed items...")

for item in failed_items:
    topic = item.get("topic", "")
    folder = Path(item["folder"])
    py_path = folder / "scene.py"
    err_path = folder / "error.txt"

    if not py_path.exists():
        print(f"Skip (missing scene.py): {folder}")
        continue

    code = py_path.read_text(encoding="utf-8")
    class_name = _extract_class_name(code) or scene_class_name(topic)

    error_text = err_path.read_text(encoding="utf-8") if err_path.exists() else item.get("error", "")

    repaired = False
    last_error = ""

    for attempt in range(1, 4):
        ok, parse_err = _is_valid_python(code)
        if not ok:
            last_error = parse_err
            code = _repair_code_with_llm(code, error_text + "\n" + parse_err, topic, class_name)
            py_path.write_text(code, encoding="utf-8")

        try:
            render_scene(str(py_path), class_name, str(folder))
            media_dir = folder / "videos"
            mp4s = sorted(media_dir.rglob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
            item["status"] = "ok"
            item["video"] = str(mp4s[0]) if mp4s else ""
            item.pop("error", None)
            repaired = True
            print(f"Re-render success: {topic}")
            break
        except Exception as e:
            last_error = str(e)
            error_text = last_error
            (folder / "error.txt").write_text(traceback.format_exc(), encoding="utf-8")
            code = _repair_code_with_llm(code, error_text, topic, class_name)
            py_path.write_text(code, encoding="utf-8")

    if not repaired:
        item["status"] = "failed"
        item["error"] = last_error
        print(f"Still failed: {topic}")

summary_path.write_text(json.dumps(items, ensure_ascii=False, indent=2), encoding="utf-8")
print("Updated summary:", summary_path)

if globals().get("DRIVE_OUTPUT_ROOT") is not None:
    drive_run_dir = DRIVE_OUTPUT_ROOT / OUTPUT_ROOT.name
    if drive_run_dir.exists():
        shutil.rmtree(drive_run_dir)
    shutil.copytree(OUTPUT_ROOT, drive_run_dir)

    zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT))
    shutil.copy2(zip_path, DRIVE_OUTPUT_ROOT / zip_path.name)

    print("Synced repaired folder to Drive:", drive_run_dir)
    print("Synced repaired zip to Drive:", DRIVE_OUTPUT_ROOT / zip_path.name)

Trying to repair 3 failed items...
Re-render success: Base-2 Number Systems and Data Representation
Re-render success: Fundamentals of Personal Cybersecurity and Defense
Re-render success: Version Control Principles via Collaborative Workflows
Updated summary: /content/education_video/manim_outputs/run_20260707_083843/summary.json
Synced repaired folder to Drive: /content/drive/MyDrive/education_video/manim_outputs/run_20260707_083843
Synced repaired zip to Drive: /content/drive/MyDrive/education_video/manim_outputs/run_20260707_083843.zip


In [26]:
# 6) Quick result view

ok = [r for r in results if r.get("status") == "ok"]
failed = [r for r in results if r.get("status") != "ok"]

print(f"Output root: {OUTPUT_ROOT}")
print(f"Success: {len(ok)} / {len(results)}")

if ok:
    print("\nRendered videos:")
    for r in ok:
        print("-", r["video"])

if failed:
    print("\nFailed topics:")
    for r in failed:
        print("-", r["topic"], "=>", r.get("error", "unknown"))

Output root: /content/education_video/manim_outputs/run_20260707_083843
Success: 0 / 3

Failed topics:
- Base-2 Number Systems and Data Representation => RuntimeError: Manim render failed (exit_code=1)
CMD: /usr/bin/python3 -m manim -ql /content/education_video/manim_outputs/run_20260707_083843/01_tm_cs_bin_001_Base-2_Number_Systems_and_Data_Representation/scene.py Base2NumberSystemsAndDataRepresentationScene --media_dir /content/education_video/manim_outputs/run_20260707_083843/01_tm_cs_bin_001_Base-2_Number_Systems_and_Data_Representation

STDOUT:
Manim Community v0.20.1



STDERR:
<frozen runpy>:128: RuntimeWarning: 'manim.__main__' found in sys.modules after import of package 'manim', but prior to execution of 'manim.__main__'; this may result in unpredictable behaviour
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/manim/__main__.py", line 10